<a href="https://colab.research.google.com/github/engmoatazmostafa/AIProject/blob/master/Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import pandas as pd
from PIL import Image
import os

# 1. Configuration
DATA_DIR = "../../../CheXpert-v1.0-small/"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
BATCH_SIZE = 2
EPOCHS = 3
NUM_WORKERS = 2
# Target cardiovascular labels: Cardiomegaly and Edema
TARGET_LABELS = ['Cardiomegaly', 'Edema']


# 2. Custom Dataset Loader
class CheXpertDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        if not os.path.isfile(csv_file):
            raise FileNotFoundError(f"CSV file not found: {csv_file}")
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        # Handle uncertainty labels: simple 'U-Ones' approach (map -1 to 1)
        self.df = self.df.fillna(0).replace(-1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_rel_path = self.df.iloc[idx]['Path']
        img_path = os.path.join(DATA_DIR, img_rel_path)
        print(f"Loading image: {img_path}")
        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image file not found: {img_path}")
        image = Image.open(img_path).convert('RGB')
        # Extract binary labels for the target cardiovascular diseases
        labels_np = self.df.iloc[idx][TARGET_LABELS].values.astype(float)
        labels = torch.tensor(labels_np, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)
        return image, labels


# 3. Preprocessing & Augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Standard for DenseNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def get_model(num_labels):
    # DenseNet-121 is the benchmark for CheXpert
    # Support both old (pretrained=True) and new (weights=...) torchvision APIs
    try:
        # Newer torchvision: weights argument
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)
    except Exception:
        # Fallback to older API
        model = models.densenet121(pretrained=True)

    # Replace the classifier with a linear layer (we'll use BCEWithLogitsLoss)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, num_labels)
    return model


def get_device():
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    print("CUDA not available, using CPU")
    return torch.device("cpu")


def build_dataloader(dataset, device):
    use_cuda = device.type == "cuda"
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=use_cuda,
        persistent_workers=NUM_WORKERS > 0,
    )


def main():
    device = get_device()
    use_cuda = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)

    # 4. Model Setup
    model = get_model(len(TARGET_LABELS))
    model = model.to(device)

    # 5. Training Loop
    train_data = CheXpertDataset(TRAIN_CSV, transform=transform)
    train_loader = build_dataloader(train_data, device)

    # Use BCEWithLogitsLoss and let the model output raw logits
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    print(f"Training on device: {device}")
    if use_cuda:
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")

    print(f"Starting training for {EPOCHS} epochs")
    for epoch in range(EPOCHS):
        print(f"Starting training epoch# {epoch + 1}")
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            print(f"Processing batch with {images.size(0)} samples")
            images = images.to(device, non_blocking=use_cuda)
            labels = labels.to(device, non_blocking=use_cuda)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_cuda):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {epoch_loss:.4f}")

    # Save the trained model
    torch.save(model.state_dict(), "cardio_chexpert_model.pth")


if __name__ == "__main__":
    main()
